# Criação e inserção de embeddings
### Fazendo os embeddings dos titulos das producoes com o modelo paraphrase-multilingual-mpnet-base-v2

Recuperando dados das producoes

In [1]:
%store -r prod_list

Importacoes necessarias + definicao do modelo

In [2]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

c:\Users\luiza\anaconda3\envs\icbiobd2025\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def init_driver(uri, username, password):
    '''Funcao que conecta ao banco do Neo4J e verifica se a conexao foi bem sucedida.'''
    driver = GraphDatabase().driver(str(uri), auth=(username, password))
    driver.verify_connectivity()
    return driver

load_dotenv()
NEO4J_USERNAME =  os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD =  os.getenv('NEO4J_PASSWORD')
NEO4J_URI = os.getenv('NEO4J_URI')

driver = init_driver(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

In [4]:
def cria_embeddings(model, prod_list, r) -> list:
    prod_emb = []
    print(f'Criando embeddings para {r} produtos...')
    for prod in prod_list[:r]:
        embedding = model.encode(prod['titulo'])
        prod_emb.append({'embedding': embedding, 'titulo': prod['titulo']})
    print(f'Embeddings criados!')
    return prod_emb

In [ ]:
def insert_embeddings(driver, list_embeddings, batch_size=1000, verbose=False):
    '''Insere os embeddings criados nos nodes da respectiva producao'''
    query = """
    UNWIND $list_embeddings AS list_emb
    MATCH (p:Producao {titulo: list_emb.titulo})
    SET p.embedding = list_emb.embedding
    RETURN p
    """
    with driver.session() as session:
        try:
            print("Inserindo embeddings dos titulos...")
            for i in range(0, len(list_embeddings), batch_size):
                batch = list_embeddings[i:i + batch_size]
                result = session.run(query, list_embeddings=batch)
                if verbose:
                    for record in result:
                        print(f"Embeddings da producao {record['p']['titulo']} inserido.")
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()
            print("Embeddings inseridos!")

In [6]:
prod_emb_list = cria_embeddings(model, prod_list, len(prod_list))
insert_embeddings(driver, prod_emb_list, 100, verbose=False)

Criando embeddings para 90906 produtos...
Embeddings criados!
Inserindo relacoes professores-departamentos...
Embeddings inseridos!
